# 01 — Stratified Sampling Frame  
### *Coding Alone* · ICSE 2027 SEIS pipeline

**Self-contained:** this notebook sets up its own auth + helpers (same as notebook 00), then builds a reproducible, stratified **~50,000-repo** OSS sample from **GH Archive**.

**How to run:** put your project ID in the config cell (the same ID that worked in notebook 00), then **Runtime → Run all**. Complete the sign-in popup when it appears.

**Fully interruption-safe.** Each year writes to its **own** table (`repo_year_agg_<yr>`, overwrite) and is recorded in a Drive manifest; re-running **skips finished years** and reattaches to any BigQuery job still running server-side. Nothing is appended blindly — no duplicates, no loss.

Steps: setup → (1) per-year aggregation → (2) lifetime summary → (3) optional language → (4) inclusion filter → (5) stratified sample → (6) save → (7) describe.


## Config — edit PROJECT_ID (use the SAME id that worked in notebook 00)

In [ ]:
# ==== EDIT ME ==============================================================
PROJECT_ID = "YOUR-GCP-PROJECT-ID"     # same ID you used in notebook 00
# ==========================================================================
WINDOW_START = "201801"       # YYYYMM
WINDOW_END   = "202606"       # YYYYMM  (latest COMPLETE month; June 2026 is complete)
BQ_DATASET   = "seis"
BQ_LOCATION  = "US"
MAX_SCAN_GB  = 1500           # per-query hard cap (~$9.4). Guards against accidents.
DRIVE_BASE   = "/content/drive/MyDrive/ICSE_SEIS"

assert PROJECT_ID != "PASTE-YOUR-PROJECT-ID-HERE", "Set PROJECT_ID first."
print("Config loaded. PROJECT_ID =", PROJECT_ID, "| window", WINDOW_START, "->", WINDOW_END)


## Setup — auth, BigQuery, Drive, dataset (same as notebook 00)
A sign-in popup appears here — choose **ANON@institution** and complete every consent screen.

In [ ]:
import os, json, time, functools
import pandas as pd, numpy as np
from google.colab import auth, drive
from google.cloud import bigquery

auth.authenticate_user()
client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
drive.mount('/content/drive')
for sub in ["", "/data", "/data/derived", "/figures", "/tables", "/checkpoints"]:
    os.makedirs(DRIVE_BASE + sub, exist_ok=True)
ds = bigquery.Dataset(f"{PROJECT_ID}.{BQ_DATASET}"); ds.location = BQ_LOCATION
client.create_dataset(ds, exists_ok=True)
print("Connected. Project:", client.project, "| dataset:", BQ_DATASET)


## Resilience helpers (checkpoints, atomic writes, BigQuery job ledger)

In [ ]:
CKPT_DIR = f"{DRIVE_BASE}/checkpoints"; os.makedirs(CKPT_DIR, exist_ok=True)

def _atomic_write_text(path, text):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        f.write(text); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)

def load_json(name, default):
    p = f"{CKPT_DIR}/{name}"
    if os.path.exists(p):
        with open(p) as f: return json.load(f)
    return default
def save_json(name, obj):
    _atomic_write_text(f"{CKPT_DIR}/{name}", json.dumps(obj, indent=2, default=str))

def manifest_done(name): return set(load_json(f"manifest_{name}.json", []))
def manifest_mark(name, key):
    s = manifest_done(name); s.add(str(key))
    save_json(f"manifest_{name}.json", sorted(s)); return s

def atomic_write_df(df, path):
    tmp = path + ".tmp"; df.to_parquet(tmp, index=False); os.replace(tmp, path)
def atomic_read_df(path):
    return pd.read_parquet(path) if os.path.exists(path) else None

_job_records = load_json("bq_jobs.json", {})
def _save_job_records(): save_json("bq_jobs.json", _job_records)

def run_query(sql, destination=None, write="WRITE_TRUNCATE", override=False,
              to_dataframe=False, label=None, job_key=None):
    tag = f"[{label}] " if label else ""
    if job_key and job_key in _job_records:
        rec = _job_records[job_key]
        try:
            prev = client.get_job(rec["job_id"], location=rec.get("location", BQ_LOCATION))
            if prev.state == "DONE" and prev.error_result is None:
                print(f"{tag}Reusing finished job {rec['job_id']} (no recompute).")
                r = prev.result(); return r.to_dataframe() if to_dataframe else r
        except Exception as e:
            print(f"{tag}(could not reattach: {e}; resubmitting)")
    dry = client.query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False))
    gb = dry.total_bytes_processed / 1e9
    print(f"{tag}Will scan ~{gb:,.1f} GB  (~${gb/1000*6.25:,.2f})")
    if gb > MAX_SCAN_GB and not override:
        raise RuntimeError(f"ABORT: {gb:,.1f} GB > MAX_SCAN_GB={MAX_SCAN_GB}. Pass override=True to force.")
    cfg = bigquery.QueryJobConfig(maximum_bytes_billed=int(MAX_SCAN_GB*1e9*1.05))
    if destination:
        cfg.destination = f"{PROJECT_ID}.{BQ_DATASET}.{destination}"; cfg.write_disposition = write
    job = client.query(sql, job_config=cfg)
    if job_key:
        _job_records[job_key] = {"job_id": job.job_id, "location": BQ_LOCATION}; _save_job_records()
    r = job.result()
    print(f"{tag}Done. Scanned {job.total_bytes_processed/1e9:,.1f} GB" + (f" -> {destination}" if destination else ""))
    return r.to_dataframe() if to_dataframe else r

YEARS = list(range(int(WINDOW_START[:4]), int(WINDOW_END[:4]) + 1))
print("Helpers ready. Years:", YEARS, "| years already done:", sorted(manifest_done('year_agg')))


## 1. Per-year, per-repo aggregation — idempotent & resumable
One bounded query per year. `WatchEvent` = stars; distinct `actor.id` ≈ contributor breadth; per-type counts feed the isolation metrics later. Each year → its own table + manifest mark. **Watch the "Will scan ~X GB" line** the helper prints before each year — that's your live cost meter.

In [ ]:
def year_agg_sql(yr):
    start = f"{yr}01"
    end   = f"{yr}12" if yr != YEARS[-1] else min(f"{yr}12", WINDOW_END)
    return f"""
    SELECT
      repo.id AS repo_id, ANY_VALUE(repo.name) AS repo_name, {yr} AS yr,
      COUNT(*) AS events, COUNT(DISTINCT actor.id) AS distinct_actors,
      COUNTIF(type='WatchEvent') AS stars_gained,
      COUNTIF(type='PushEvent') AS push_events,
      COUNTIF(type='PullRequestEvent') AS pr_events,
      COUNTIF(type='IssuesEvent') AS issue_events,
      COUNTIF(type='IssueCommentEvent') AS issue_comment_events,
      COUNTIF(type='PullRequestReviewCommentEvent') AS review_comment_events,
      COUNTIF(type='PullRequestReviewEvent') AS review_events,
      COUNTIF(type='ForkEvent') AS fork_events,
      MIN(created_at) AS first_seen, MAX(created_at) AS last_seen
    FROM `githubarchive.month.*`
    WHERE _TABLE_SUFFIX BETWEEN '{start}' AND '{end}' AND repo.id IS NOT NULL
    GROUP BY repo_id
    """

for yr in YEARS:
    if str(yr) in manifest_done("year_agg"):
        print(f"year {yr}: already complete — skip"); continue
    run_query(year_agg_sql(yr), destination=f"repo_year_agg_{yr}",
              write="WRITE_TRUNCATE", label=f"year-agg-{yr}", job_key=f"year_agg_{yr}")
    manifest_mark("year_agg", yr)
    print(f"year {yr}: done & checkpointed")
print("All years complete:", sorted(manifest_done('year_agg')))


## 2. Lifetime summary
Rolls the per-year tables up to one row per repo. `peak_year_actors` is the contributor **stratification proxy**; exact counts come in the Tier-B deep fetch.

In [ ]:
lifetime_sql = f"""
SELECT repo_id, ANY_VALUE(repo_name) AS repo_name,
  SUM(events) AS total_events, SUM(stars_gained) AS total_stars,
  SUM(push_events) AS total_pushes, SUM(pr_events) AS total_pr,
  SUM(issue_events) AS total_issues, SUM(issue_comment_events) AS total_issue_comments,
  SUM(review_comment_events) AS total_review_comments, SUM(review_events) AS total_reviews,
  SUM(fork_events) AS total_forks, MAX(distinct_actors) AS peak_year_actors,
  MIN(first_seen) AS first_seen, MAX(last_seen) AS last_seen,
  DATE_DIFF(DATE(MAX(last_seen)), DATE(MIN(first_seen)), DAY) AS age_days,
  COUNT(DISTINCT yr) AS active_years
FROM `{PROJECT_ID}.{BQ_DATASET}.repo_year_agg_*`
GROUP BY repo_id
"""
run_query(lifetime_sql, destination="repo_lifetime", label="lifetime", job_key="lifetime")
print("repo_lifetime built.")


## 3. (Optional) primary language
LEFT JOIN to the public `github_repos.languages` snapshot; unmatched → `unknown`. Skippable.

In [ ]:
lang_sql = f"""
WITH top_lang AS (
  SELECT repo_name,
    (SELECT l.name FROM UNNEST(language) l ORDER BY l.bytes DESC LIMIT 1) AS primary_language
  FROM `bigquery-public-data.github_repos.languages`)
SELECT r.*, COALESCE(t.primary_language,'unknown') AS primary_language
FROM `{PROJECT_ID}.{BQ_DATASET}.repo_lifetime` r
LEFT JOIN top_lang t ON LOWER(r.repo_name)=LOWER(t.repo_name)
"""
run_query(lang_sql, destination="repo_lifetime_lang", label="language-join", job_key="lang")
print("repo_lifetime_lang built.")


## 4. Inclusion filter → candidates (checkpointed to Drive)
Keep plausibly-real collaborative OSS repos; save candidates to Drive so step 5 resumes without re-querying.

In [ ]:
MIN_EVENTS, MIN_AGE_DAYS, MIN_ACTIVE_YRS, REQUIRE_COLLAB = 50, 90, 1, True
src = f"{PROJECT_ID}.{BQ_DATASET}.repo_lifetime_lang"   # or .repo_lifetime if you skipped step 3
collab = "AND (total_pr + total_issues) >= 1" if REQUIRE_COLLAB else ""

cand_path = f"{DRIVE_BASE}/data/derived/candidates.parquet"
cand = atomic_read_df(cand_path)
if cand is None:
    cand = run_query(f"""
      SELECT * FROM `{src}`
      WHERE total_events >= {MIN_EVENTS} AND age_days >= {MIN_AGE_DAYS}
        AND active_years >= {MIN_ACTIVE_YRS} {collab}
    """, to_dataframe=True, label="candidates", job_key="candidates")
    atomic_write_df(cand, cand_path)
print(f"{len(cand):,} candidate repos (checkpoint: {cand_path})")
cand.head()


## 5. Stratified sample → ~50k (fixed seed = reproducible)
Bin on stars × contributors × activity × age (× language if present); proportional allocation with a floor so rare-but-important cells survive.

In [ ]:
TARGET_N = 50_000
df = cand.copy()
def qbin(s, k=4):
    try:    return pd.qcut(s.rank(method="first"), k, labels=[f"q{i+1}" for i in range(k)])
    except Exception: return pd.cut(s, k, labels=[f"q{i+1}" for i in range(k)])
df["star_bin"]=qbin(df.total_stars); df["contrib_bin"]=qbin(df.peak_year_actors)
df["act_bin"]=qbin(df.total_events); df["age_bin"]=qbin(df.age_days)
strata = ["star_bin","contrib_bin","act_bin","age_bin"]
if "primary_language" in df.columns:
    top = df.primary_language.value_counts().head(12).index
    df["lang_bin"] = np.where(df.primary_language.isin(top), df.primary_language, "other")
    strata.append("lang_bin")

grp = df.groupby(strata, observed=True)
frac = min(1.0, TARGET_N/len(df)); FLOOR=1
frame = grp.apply(lambda g: g.sample(n=min(len(g), max(FLOOR, round(len(g)*frac))),
                                     random_state=42)).reset_index(drop=True)
if len(frame) > TARGET_N: frame = frame.sample(n=TARGET_N, random_state=7)
print(f"Frame: {len(frame):,} repos across {grp.ngroups:,} strata (target {TARGET_N:,})")
frame[["repo_id","repo_name","total_stars","peak_year_actors","total_events","age_days"]].head()


## 6. Save frame — BigQuery (for JOINs) + Drive (portable, atomic)

In [ ]:
job = client.load_table_from_dataframe(
    frame, f"{PROJECT_ID}.{BQ_DATASET}.sampling_frame",
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"))
job.result(); print("Wrote seis.sampling_frame:", len(frame), "rows")
atomic_write_df(frame, f"{DRIVE_BASE}/data/derived/sampling_frame.parquet")
manifest_mark("stages", "01_sampling_frame")
print("Saved frame to Drive + marked stage complete.")


## 7. Frame description (for the paper's Dataset section)

In [ ]:
desc = {"n_repos": len(frame),
        "stars_median": int(frame.total_stars.median()),
        "contrib_median": float(frame.peak_year_actors.median()),
        "events_median": int(frame.total_events.median()),
        "age_days_median": int(frame.age_days.median()),
        "first_seen_min": str(frame.first_seen.min()),
        "last_seen_max": str(frame.last_seen.max())}
print(json.dumps(desc, indent=2))
if "primary_language" in frame.columns: display(frame.primary_language.value_counts().head(15))
pd.DataFrame([desc]).to_csv(f"{DRIVE_BASE}/tables/frame_description.csv", index=False)
print("Saved frame_description.csv")


## Done ✅
Reproducible, stratified **~50k-repo frame** in `seis.sampling_frame` + on Drive.
Resumable: re-running skips completed years/jobs and reloads checkpoints — no data lost or duplicated.

Next: **`02_population_events.ipynb`** — the repo-month event panel feeding the isolation metrics.